# Módulo B.1 — Preparación y validación cruzada espacial

Prepara los datos reales para el modelado: separa las variables X e y y establece la validación cruzada espacial, que ahora es relevante, incluyendo las presencias globales.

## 1. Montar Drive y cargar el dataset enriquecido

In [1]:
import os
from pathlib import Path
import pandas as pd

if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')

CARPETA_BASE = Path('/content/drive/MyDrive/TFM_TeaSuitability')
RUTA_CSV = CARPETA_BASE / 'dataset_master_enriquecido.csv'

if not RUTA_CSV.exists():
    print('No encuentro:', RUTA_CSV)
    for e in sorted(CARPETA_BASE.glob('*')):
        print('  -', e.name)
    raise FileNotFoundError('Falta el enriquecido. Ejecuta el A.2.')

dataset = pd.read_csv(RUTA_CSV)
print('Cargado. Forma:', dataset.shape)
dataset.head()

Mounted at /content/drive
Cargado. Forma: (1333, 14)


,lon,lat,clase,temperatura_media,rango_diurno,precipitacion_anual,estacionalidad_precip,precip_trimestre_seco,elevacion,ph_suelo,pca_1,pca_2,cluster_kmeans,cluster_dbscan
0,118.013054,29.941719,1,14.924833,7.618333,1670.0,54.601158,158.0,429.0,5.4,0.621109,-0.320089,2,0
1,47.241323,-20.521527,1,16.620459,11.176250,1437.0,87.080925,72.0,1472.0,5.4,-0.861451,-0.371881,2,0
2,121.503180,25.163700,1,20.207001,5.704167,3005.0,26.002686,554.0,256.0,4.7,3.501824,-0.299499,0,0
3,121.741242,24.805763,1,20.652042,5.553583,3160.0,35.268108,509.0,297.0,4.9,3.280783,-0.102303,0,0
4,120.603078,23.564246,1,19.789417,5.974333,2526.0,91.668259,76.0,748.0,5.2,0.761332,0.530576,2,0


## 2. Importaciones

In [2]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold, train_test_split

## 3. Variables predictoras

In [3]:
# Predictoras = las 8 variables ambientales (fuera lon, lat, clase y las
# columnas derivadas de A.2: pca_* y cluster_*)
columnas_predictoras = [
    'temperatura_media',        # bio1: temperatura media anual (óptimo del té: 18-25 °C)
    'rango_diurno',             # bio2: diferencia entre la máxima del día y la mínima nocturna
    'precipitacion_anual',      # bio12: lluvia total anual (óptimo 1500-3000 mm)
    'estacionalidad_precip',    # bio15: variación de la lluvia entre estaciones
    'precip_trimestre_seco',    # bio17: lluvia en la estación seca (estrés hídrico)
    'elevacion',                # altitud del terreno (el té se cultiva hasta ~2200 m)
    'ph_suelo',                 # pH del suelo (SoilGrids); el té prefiere ácido, 4.5-5.5
]

## 4. Funciones de preparación y validación

In [4]:
def separar_x_y(dataframe, columnas_predictoras, columna_objetivo='clase'):
    """
    Separa las variables predictoras (X) de la variable objetivo (y).

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Tabla completa con predictoras y objetivo.
    columnas_predictoras : list of str
        Nombres de las variables que usará el modelo.
    columna_objetivo : str, optional
        Nombre de la columna a predecir. Por defecto 'clase'.

    Returns
    -------
    pandas.DataFrame
        X, las variables predictoras.
    pandas.Series
        y, la variable objetivo.
    """
    X = dataframe[columnas_predictoras]
    y = dataframe[columna_objetivo]
    return X, y


def crear_bloques_espaciales(dataframe, n_bloques=5, semilla=42):
    """
    Agrupa los puntos por cercanía geográfica en bloques espaciales.

    Usa K-Means sobre las coordenadas (lon, lat). Cada bloque es una región
    que se mantendrá unida en la validación, para evitar la fuga espacial.

    Parameters
    ----------
    dataframe : pandas.DataFrame
        Tabla con columnas 'lon' y 'lat'.
    n_bloques : int, optional
        Número de bloques (regiones) a formar. Por defecto 5.
    semilla : int, optional
        Semilla aleatoria. Por defecto 42.

    Returns
    -------
    numpy.ndarray
        Número de bloque (0..n_bloques-1) de cada punto.
    """
    coordenadas = dataframe[['lon', 'lat']]
    agrupador = KMeans(n_clusters=n_bloques, random_state=semilla, n_init=10)
    return agrupador.fit_predict(coordenadas)


def resumir_folds(X, y, bloques):
    """
    Muestra cómo queda cada pliegue de la validación cruzada espacial.

    Parameters
    ----------
    X : pandas.DataFrame
        Variables predictoras.
    y : pandas.Series
        Variable objetivo.
    bloques : numpy.ndarray
        Bloque espacial de cada punto.

    Returns
    -------
    sklearn.model_selection.GroupKFold
        El validador espacial configurado.
    """
    n = len(set(bloques))
    validador = GroupKFold(n_splits=n)
    for i, (tr, te) in enumerate(validador.split(X, y, groups=bloques), 1):
        pos = int(y.iloc[te].sum())
        print(f'Fold {i}: prueba={len(te)}, positivos en prueba={pos}')
    return validador

## 5. Ejecución: separar X e y

In [5]:
X, y = separar_x_y(dataset, columnas_predictoras)
print('X:', X.shape, '| y:', y.shape)
print('Distribución de clases:')
print(y.value_counts())

X: (1333, 7) | y: (1333,)
Distribución de clases:
clase
1    780
0    553
Name: count, dtype: int64


## 6. Ejecución: validación cruzada espacial

In [6]:
bloques_espaciales = crear_bloques_espaciales(dataset, n_bloques=5)
print('Tamaño de cada bloque:')
print(pd.Series(bloques_espaciales).value_counts().sort_index())

print('\nReparto de positivos por pliegue (validación espacial):')
validacion_espacial = resumir_folds(X, y, bloques_espaciales)

Tamaño de cada bloque:
0    557
1    198
2    185
3    223
4    170
Name: count, dtype: int64

Reparto de positivos por pliegue (validación espacial):
Fold 1: prueba=557, positivos en prueba=449
Fold 2: prueba=223, positivos en prueba=119
Fold 3: prueba=198, positivos en prueba=150
Fold 4: prueba=185, positivos en prueba=30
Fold 5: prueba=170, positivos en prueba=32


## 7. Ejecución: división estratificada (para entrenar/evaluar)

La validación espacial se basa en datos reales y es nuestra medida principal. También se ha preparado una división estratificada del 70/30 para el entrenamiento directo.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)
print('Entrenamiento:', X_train.shape[0], 'filas,', int(y_train.sum()), 'positivos')
print('Prueba:       ', X_test.shape[0], 'filas,', int(y_test.sum()), 'positivos')

Entrenamiento: 933 filas, 546 positivos
Prueba:        400 filas, 234 positivos


## 8. Resumen de la preparación

In [8]:
print('Preparación completada:')
print(f'  - X: {X.shape[0]} filas, {X.shape[1]} variables predictoras')
print(f'  - y: {int(y.sum())} positivos de {len(y)}')
print(f'  - Validación espacial: GroupKFold con 5 bloques (todos con positivos)')
print(f'  - División: estratificada 70/30')

Preparación completada:
  - X: 1333 filas, 7 variables predictoras
  - y: 780 positivos de 1333
  - Validación espacial: GroupKFold con 5 bloques (todos con positivos)
  - División: estratificada 70/30
